In [1]:
import os
from huggingface_hub import hf_hub_download
import duckdb
import pandas as pd

# download
path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

con = duckdb.connect()

# Aggregate to page-level: first half vs second half of March
df = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_avg_position,
            gsc_clicks,
            CASE WHEN report_date <= DATE '2026-03-15' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{path}')
        WHERE gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT
            content_hash_id,
            client_hash_id,
            period,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM daily
        GROUP BY 1,2,3
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.impressions AS impressions_first,
        s.impressions AS impressions_second,
        f.avg_position AS avg_position_first,
        s.avg_position AS avg_position_second
    FROM agg f
    JOIN agg s
      ON f.content_hash_id = s.content_hash_id
     AND f.client_hash_id = s.client_hash_id
    WHERE f.period = 'first_half' AND s.period = 'second_half'
""").df()

df.shape

(141467, 6)

In [2]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))

train = df.iloc[train_idx]
test = df.iloc[test_idx]

print(train.shape, test.shape)
print("Unique clients in train:", train["client_hash_id"].nunique())
print("Unique clients in test:", test["client_hash_id"].nunique())

(131220, 6) (10247, 6)
Unique clients in train: 34
Unique clients in test: 9


In [3]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))

train = df.iloc[train_idx]
test = df.iloc[test_idx]

print(train.shape, test.shape)
print(train.columns.tolist())

(131220, 6) (10247, 6)
['content_hash_id', 'client_hash_id', 'impressions_first', 'impressions_second', 'avg_position_first', 'avg_position_second']


In [6]:
df["pct_change_impressions"] = (
    (df["impressions_second"] - df["impressions_first"]) / df["impressions_first"].replace(0, pd.NA)
)
df["is_declining"] = (df["pct_change_impressions"] < -0.10).astype(int)

df.columns.tolist()

['content_hash_id',
 'client_hash_id',
 'impressions_first',
 'impressions_second',
 'avg_position_first',
 'avg_position_second',
 'pct_change_impressions',
 'is_declining']

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

feature_cols = ["impressions_first", "avg_position_first"]

# Naive random split — ignores client_hash_id entirely
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    df[feature_cols], df["is_declining"], test_size=0.2, random_state=42
)

tree_rand = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_rand.fit(X_train_rand, y_train_rand)
rand_probs = tree_rand.predict_proba(X_test_rand)[:, 1]
rand_auc = roc_auc_score(y_test_rand, rand_probs)

print("Random split AUC:", round(rand_auc, 4))

Random split AUC: 0.5578


In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))

train = df.iloc[train_idx]
test = df.iloc[test_idx]

X_train, y_train = train[feature_cols], train["is_declining"]
X_test, y_test = test[feature_cols], test["is_declining"]

tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)
tree_probs = tree.predict_proba(X_test)[:, 1]
tree_auc = roc_auc_score(y_test, tree_probs)

print("Grouped split AUC:", round(tree_auc, 4))

Grouped split AUC: 0.5511


In [9]:
train_clients_rand = set(df.iloc[X_train_rand.index]["client_hash_id"])
test_clients_rand = set(df.iloc[X_test_rand.index]["client_hash_id"])
overlap_rand = train_clients_rand & test_clients_rand

print("Random split — clients in both train and test:", len(overlap_rand))
print("Grouped split — clients in both train and test:", len(set(train["client_hash_id"]) & set(test["client_hash_id"])))

Random split — clients in both train and test: 41
Grouped split — clients in both train and test: 0


- Random split (naive): AUC = 0.5578 with 41 clients appearing in both train and test so it means the test contain pages from clients the model already trained on.

- Grouped split AUC = 0.5511 with zero client overlap, a true test on unseen clients.

- Interpretation: The gap between the two (0.5578 vs 0.5511) is small about 0.007 AUC. This is smaller than I expected and for this specific model and feature set, client identity does not appear to be do much of the work. The 2 features i'm using are "impressions_first" and "avg_position_first" seem to generalize similarity whether or not the model seen client before.

- I want to be careful not to overclaim this way. A small gap on one random split doesn't prove client leakage isn't a risk in general with only 9 test clients in grouped split. This comparasion has a lot of variance and different random seed could show difference gap and the grouped split is still the more defensible choice going forward since it's only guarantees no leakage by construction even this particular shows a small effect.

Leakage Audit

This secrion we will go through each feature used in the model to check its leak information or not from the label or from the future (second half of March):

- "impressions_first": Sum of GSC impressions from the first half of March only (2026-03-01 to 2026-03-15). Its computed entirely before the decision point and not derived from "is_declining"
=> Safe

- "avg_position_first": Average GSC position from the first half of March only. Same window as above, same reasoning.
=> Safe

- "is_declining": Derived Derived from comparing "impressions_second" to "impressions_first" and I confirm that this is not include as a feature anywhere in training ("feature_cols" only contains "impressions_first" and "avg_position_first")

- "impressions_second", "avg_position_second", "pct_change_impressions": All second half of the March and I confirm that these are available in df for label construction but exculde from "feature_cols" and never pass to X_train and X_test

- "client_hash_id": Used only for grouping the split and never as a model feature. This avoids the model learning as shortcut instead of learning from page-level signals.

=> Conclusion: No label-derived or future-window fields are used as features. The two features used are both computed strictly from the first half of the observation window, consistent with the discipline established in Week 3.

In [10]:
print("Features used in training:", feature_cols)
print("Label column:", "is_declining")
print("Any overlap between features and label-derivation columns?",
      bool(set(feature_cols) & {"impressions_second", "avg_position_second", "pct_change_impressions", "is_declining"}))

Features used in training: ['impressions_first', 'avg_position_first']
Label column: is_declining
Any overlap between features and label-derivation columns? False


In [11]:
# Rebuild test_results using the grouped split (the honest one)
test_results = test.copy()
test_results["tree_prob"] = tree_probs
test_results["tree_pred"] = (test_results["tree_prob"] >= 0.5).astype(int)

# False negatives: actually declining, predicted stable
false_negatives = test_results[
    (test_results["is_declining"] == 1) & (test_results["tree_pred"] == 0)
]

# Look at a few real examples
false_negatives[[
    "content_hash_id", "client_hash_id",
    "impressions_first", "impressions_second",
    "avg_position_first", "avg_position_second",
    "tree_prob"
]].sample(5, random_state=42)

,content_hash_id,client_hash_id,impressions_first,impressions_second,avg_position_first,avg_position_second,tree_prob
15850,content_3c40701baf097820,client_e5c2aa26a8598242,255.0,91.0,2.677703,9.715179,0.371128
40678,content_7dbd51d4c93d39fc,client_e5c2aa26a8598242,254.0,199.0,2.655665,2.465053,0.371128
69967,content_9c7720d9c3c8904f,client_3f0ce4d44fe94f3d,15.0,7.0,2.937500,8.000000,0.371128
11795,content_845681ddeb65af52,client_e5c2aa26a8598242,901.0,717.0,14.088166,19.481146,0.343046
115727,content_8075de41cb32d132,client_3f0ce4d44fe94f3d,8.0,4.0,37.000000,4.000000,0.343046


## Real Failure Examples

Five actual false negatives pages that declined but the model predicted stable:

- "content_3c40701baf097820": impressions drop from 255 -> 91, it was real sharp decline. Model probability was 0.371 and it was below thresholding. Likely missed because 255 impressions still 
sits in a "mid-range" bucket.

- "content_9c7720d9c3c8904f": impressions drop from 15 -> 7, a very low volume overall and this is close to the noise floor where a handful of impressions either way can flip the percentage change without reflecting a real trend and the model likely correctly treats low volume pages as low confidence. 

- "content_8075de41cb32d132": impressions drop from 8 -> 4, an average position actually improved sharply from 37 -> 4. This page's signals point in conflicting directions and it was worse on my decline label but better on position which can be case of 2 feature model can't resolve cleanly.

=> Pattern accross case: all five have tree probabilities just below the 0.5 threshold (0.34–0.37). This show that model pick some signal but the fixed 0.5 threshold is excluding real declines. Reinforcing the week 5 finding that ranked, threshold free approach(like baseline queue) may be more useful in practice than a hard classfication cutoff.

## Claim-Language Audit

Reviewing claims from Weeks 1-5 for overclaiming. Most were already 
appropriately hedged (Week 1's position finding, Week 5's model 
comparison, Week 6's split comparison). The one fix: Week 4's "CONFIRMED" 
verdict language was tightened to be explicit that it reflects an 
observed, same-sample association, not a causal or generalizable claim.

Going forward, every model or signal claim uses observed/measured/
directional language, and every output is framed as decision-support for 
a human reviewer, not an automated verdict.


## Self-Check

- Two paper findings + methodology questions: [pending — need the paper]

- Own model re-run under honest split, with before/after: done — random 
  split (0.5578, full client overlap) vs. grouped split (0.5511, zero 
  overlap)

- Leakage audit: done, confirmed no label-derived or future-window 
  fields used as features

- Real failure examples: done, 5 false negatives examined, pattern 
  identified (near-threshold probabilities, low-volume noise cases, 
  conflicting-signal cases)
  
- Safe claim language throughout: done, audited and tightened